In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
print("所有库导入成功！")

In [ ]:
# 1. 加载数据
digits = load_digits()
X = digits.data
y = digits.target
print(f"样本数: {X.shape[0]}, 特征维度: {X.shape[1]}")
print(f"类别数: {len(np.unique(y))}, 类别标签: {np.unique(y)}")

In [ ]:
# 2. 数据可视化 — 展示部分数字样本
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i in range(10):
    idx = np.where(y == i)[0][0]
    axes[0, i].imshow(X[idx].reshape(8, 8), cmap='gray')
    axes[0, i].set_title(f'{i}')
    axes[0, i].axis('off')
    idx2 = np.where(y == i)[0][1]
    axes[1, i].imshow(X[idx2].reshape(8, 8), cmap='gray')
    axes[1, i].axis('off')
fig.suptitle('数字样本展示（每个数字2个样本）', fontsize=14)
plt.tight_layout()
plt.savefig('sample_digits.png', dpi=150, bbox_inches='tight')
plt.show()

# 类别分布
fig, ax = plt.subplots(figsize=(8, 4))
counts = np.bincount(y)
ax.bar(range(10), counts, color='steelblue', edgecolor='black')
ax.set_xlabel('数字类别')
ax.set_ylabel('样本数量')
ax.set_title('数据集类别分布')
ax.set_xticks(range(10))
for i, c in enumerate(counts):
    ax.text(i, c + 2, str(c), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"各类别样本数: {counts}")

In [ ]:
# 3. 数据预处理
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f"标准化后均值范围: [{X_scaled.mean(axis=0).min():.4f}, {X_scaled.mean(axis=0).max():.4f}]")
print(f"标准化后方差范围: [{X_scaled.var(axis=0).min():.4f}, {X_scaled.var(axis=0).max():.4f}]")

In [ ]:
# 4. PCA 方差解释率分析
pca_full = PCA().fit(X_scaled)
cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(range(1, 65), pca_full.explained_variance_ratio_, alpha=0.6, label='单个方差贡献')
ax1.set_xlabel('主成分编号')
ax1.set_ylabel('方差贡献率')
ax1.set_title('PCA 方差解释率分析')

ax2 = ax1.twinx()
ax2.plot(range(1, 65), cumulative_var, 'r-o', markersize=3, label='累计方差贡献率')
ax2.axhline(y=0.95, color='gray', linestyle='--', alpha=0.7, label='95% 阈值')
ax2.set_ylabel('累计方差贡献率')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')
plt.tight_layout()
plt.savefig('pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

n_95 = np.argmax(cumulative_var >= 0.95) + 1
print(f"保留95%方差需要的主成分数: {n_95}")
print(f"当前选择30维时的累计方差贡献率: {cumulative_var[29]:.4f}")

In [ ]:
# 5. PCA 降维
pca = PCA(n_components=30)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA降维后特征维度: {X_pca.shape[1]}")

In [ ]:
# 6. 超参数调优 — 搜索最优 K 值
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid_knn = {'n_neighbors': list(range(1, 21))}
grid_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn, cv=cv, scoring='accuracy', return_train_score=True)
grid_knn.fit(X_pca, y)

print(f"最优 K 值: {grid_knn.best_params_['n_neighbors']}")
print(f"最优交叉验证准确率: {grid_knn.best_score_:.4f}")

# 可视化不同K值的准确率
fig, ax = plt.subplots(figsize=(10, 5))
k_values = list(range(1, 21))
train_scores = grid_knn.cv_results_['mean_train_score']
test_scores = grid_knn.cv_results_['mean_test_score']
ax.plot(k_values, train_scores, 'b-o', label='训练集准确率')
ax.plot(k_values, test_scores, 'r-o', label='验证集准确率')
ax.axvline(x=grid_knn.best_params_['n_neighbors'], color='gray', linestyle='--', alpha=0.7)
ax.set_xlabel('K 值')
ax.set_ylabel('准确率')
ax.set_title('KNN 不同 K 值的交叉验证准确率')
ax.legend()
ax.set_xticks(k_values)
plt.tight_layout()
plt.savefig('knn_k_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 7. 超参数调优 — 搜索最优 PCA 维度
pca_dims = [10, 15, 20, 25, 30, 35, 40, 50, 64]
best_k = grid_knn.best_params_['n_neighbors']
pca_results = []

for dim in pca_dims:
    if dim == 64:
        X_dim = X_scaled
    else:
        X_dim = PCA(n_components=dim).fit_transform(X_scaled)
    knn = KNeighborsClassifier(n_neighbors=best_k)
    scores = cross_val_score(knn, X_dim, y, cv=cv, scoring='accuracy')
    pca_results.append(scores.mean())
    print(f"PCA维度={dim:2d}: 平均准确率={scores.mean():.4f} (±{scores.std():.4f})")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(pca_dims, pca_results, 'g-o', markersize=8)
ax.axvline(x=pca_dims[np.argmax(pca_results)], color='red', linestyle='--', alpha=0.7)
ax.set_xlabel('PCA 维度')
ax.set_ylabel('平均准确率')
ax.set_title('不同 PCA 维度对 KNN 分类准确率的影响')
ax.set_xticks(pca_dims)
plt.tight_layout()
plt.savefig('pca_dim_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

best_dim = pca_dims[np.argmax(pca_results)]
print(f"\n最优 PCA 维度: {best_dim}, 最优 K 值: {best_k}")

In [ ]:
# 8. 使用最优参数进行特征提取
if best_dim < 64:
    pca_best = PCA(n_components=best_dim)
    X_final = pca_best.fit_transform(X_scaled)
    print(f"应用 PCA 降维: {X_scaled.shape[1]} → {best_dim} 维")
else:
    X_final = X_scaled
    print(f"最优方案：不使用 PCA（保留全部 {X_scaled.shape[1]} 维特征）")

In [ ]:
# 9. 多分类器对比
classifiers = {
    'KNN (最优K)': KNeighborsClassifier(n_neighbors=best_k),
    'SVM': SVC(kernel='rbf', random_state=42),
    '随机森林': RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}
for name, clf in classifiers.items():
    scores = cross_val_score(clf, X_final, y, cv=cv, scoring='accuracy')
    results[name] = scores
    print(f"{name:15s}: 平均准确率={scores.mean():.4f} (±{scores.std():.4f})")

# 可视化对比
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results.keys())
means = [results[n].mean() for n in names]
stds = [results[n].std() for n in names]
bars = ax.bar(names, means, yerr=stds, capsize=8, color=['steelblue', 'coral', 'forestgreen'], edgecolor='black')
ax.set_ylabel('准确率')
ax.set_title('不同分类器 5 折交叉验证准确率对比')
ax.set_ylim(0.95, 1.0)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
            f'{mean:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('classifier_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 10. 混淆矩阵（使用最优 KNN 模型）
from sklearn.model_selection import cross_val_predict

best_clf = KNeighborsClassifier(n_neighbors=best_k)
y_pred = cross_val_predict(best_clf, X_final, y, cv=cv)

cm = confusion_matrix(y, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=digits.target_names)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('KNN 混淆矩阵（5折交叉验证预测）')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# 计算每个类别的准确率
per_class_acc = cm.diagonal() / cm.sum(axis=1)
for i, acc in enumerate(per_class_acc):
    print(f"数字 {i}: 准确率={acc:.4f}")

In [ ]:
# 11. 分类报告
print("分类报告:")
print(classification_report(y, y_pred, target_names=[f'数字{i}' for i in range(10)]))

In [ ]:
# 12. 错误样本分析
misclassified = np.where(y != y_pred)[0]
print(f"总样本数: {len(y)}")
print(f"预测错误数: {len(misclassified)}")
print(f"错误率: {len(misclassified) / len(y) * 100:.2f}%")

# 展示部分错误样本
n_show = min(10, len(misclassified))
if n_show > 0:
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for i in range(n_show):
        idx = misclassified[i]
        ax = axes[i // 5, i % 5]
        ax.imshow(X[idx].reshape(8, 8), cmap='gray')
        ax.set_title(f'真实:{y[idx]} 预测:{y_pred[idx]}', fontsize=10, color='red')
        ax.axis('off')
    # 隐藏多余子图
    for i in range(n_show, 10):
        axes[i // 5, i % 5].axis('off')
    fig.suptitle(f'错误分类样本展示（共{len(misclassified)}个错误）', fontsize=13)
    plt.tight_layout()
    plt.savefig('misclassified_samples.png', dpi=150, bbox_inches='tight')
    plt.show()

# 最常见的混淆对
print("\n最常见的混淆对（真实→预测: 次数）:")
for i in range(10):
    for j in range(10):
        if i != j and cm[i][j] > 0:
            print(f"  {i}→{j}: {cm[i][j]}次")

In [ ]:
# 13. 实验总结
print("=" * 50)
print("实验总结")
print("=" * 50)
print(f"数据集: sklearn digits (样本数={len(y)}, 原始维度=64)")
print(f"最优超参数: K={best_k}, PCA维度={best_dim}")
print(f"KNN 最优准确率: {grid_knn.best_score_:.4f}")
print()
print("分类器对比:")
for name, scores in results.items():
    print(f"  {name}: {scores.mean():.4f} (±{scores.std():.4f})")
print()
best_name = max(results, key=lambda k: results[k].mean())
print(f"最优分类器: {best_name} (准确率={results[best_name].mean():.4f})")
print(f"总错误样本数: {len(misclassified)}/{len(y)}")
print("=" * 50)